In [10]:
import polars as pl

df = pl.read_parquet(
    "datasets/megascale_dataset_with_ddg.parquet",
)
df

original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type
str,str,f64,f64,f64,str
"""GGITGDVSAANKDAIRKQMDAAASKGDVET…","""GGITGDVSAANKDAIRKQMDAAASKGDVET…",-0.691482,1.16441,-1.855892,"""Y31P"""
"""SAGGSEFTQISGYVNAFGSQRGSVLTVKVE…","""SAGGSEFTQISGYVNAFGSQRGSVHTVKVE…",-4.630889,1.703911,-6.3348,"""L25H:V57P"""
"""SAGGSAGGSALRDDEYDEWQDIIRDWRKEM…","""SAGGSAGGSALRDDEYDLWQDIIRDWRKEM…",0.262727,3.908905,-3.646178,"""E18L:Y54D"""
"""SAGGSPEVQIAILTEQINNLNEHLRVHKKD…","""SAGGSPQVQIAILTEQINNLNEHLRVHKKD…",1.932703,2.674476,-0.741773,"""E7Q"""
"""GQTIHNGDQTYHYDNPEEAIKVAQKLAKIY…","""GQTIHNIDQTYHYDNPEEAIKVAQKLAKIY…",1.46799,1.529946,-0.061956,"""G7I"""
…,…,…,…,…,…
"""SAGGNQASVVANTLIPINTALTLVMMRSEV…","""SAGGNQASVVANTLIPINTALTLVMMRSEV…",5.973451,6.20524,-0.231789,"""S46W"""
"""SAGGSAGTGIVNVSSSLNVRSSASTSSKVI…","""SAGGSAGTGIVNVSSSLNVRSSASTSSSVI…",2.993401,3.155036,-0.161636,"""K28S"""
"""GGITGDVSAANKDAIRKQMDAAASKGDVET…","""GGITGDVSAANKDAIRKQMDAAASKGDVET…",1.23246,1.16441,0.06805,"""K39R"""


In [11]:
df.select(pl.col("ddG")).describe()

statistic,ddG
str,f64
"""count""",416914.0
"""null_count""",0.0
"""mean""",-1.51392
"""std""",1.939565
"""min""",-17.674774
"""25%""",-2.245559
"""50%""",-0.92614
"""75%""",-0.177381
"""max""",8.677801


In [20]:
k_neg = 0.230
k_pos = 0.576
A_neg = 1.0
A_pos = 1.0

sigmoid_expr = (
    pl.when(pl.col("ddG") >= 0)
    .then(A_pos * (2 / (1 + (-k_pos * pl.col("ddG")).exp()) - 1))
    .otherwise(-A_neg * (2 / (1 + (-k_neg * (-pl.col("ddG"))).exp()) - 1))
)

df = df.with_columns(
    sigmoid_expr.alias("normalized_ddG_sigmoid")
)

df = df.with_columns(
    (pl.col("normalized_ddG_sigmoid")*-1) .alias("normalized_stability")
)

## add reverse mutations

# reverse mutation
df_joined = df.with_columns(pl.lit(False).alias("reverse"))

# Reverzní mutace
df_joined_with_reverse = df_joined.with_columns([
    pl.col("original_seq_full").alias("mutated_seq_full"),
    pl.col("mutated_seq_full").alias("original_seq_full"),
    # multiply by -1
    pl.col("normalized_stability") * -1,

    pl.lit(True).alias("reverse")
])




df_joined_with_reverse = pl.concat([df_joined, df_joined_with_reverse])

df_joined_with_reverse = df_joined_with_reverse.select([pl.col("original_seq_full"),
                                                       pl.col("mutated_seq_full"),
                                                       pl.col("deltaG"),
                                                       pl.col("deltaG_wt"),
                                                       pl.col("ddG"),
                                                       pl.col("mut_type"),
                                                           pl.col("normalized_stability"),
                                                       pl.col("reverse")])



df_joined_with_reverse.write_csv("datasets/megascale_dataset_with_ddg_normalized.csv")
df_joined_with_reverse

original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type,normalized_stability,reverse
str,str,f64,f64,f64,str,f64,bool
"""GGITGDVSAANKDAIRKQMDAAASKGDVET…","""GGITGDVSAANKDAIRKQMDAAASKGDVET…",-0.691482,1.16441,-1.855892,"""Y31P""",0.210245,false
"""SAGGSEFTQISGYVNAFGSQRGSVLTVKVE…","""SAGGSEFTQISGYVNAFGSQRGSVHTVKVE…",-4.630889,1.703911,-6.3348,"""L25H:V57P""",0.622148,false
"""SAGGSAGGSALRDDEYDEWQDIIRDWRKEM…","""SAGGSAGGSALRDDEYDLWQDIIRDWRKEM…",0.262727,3.908905,-3.646178,"""E18L:Y54D""",0.396349,false
"""SAGGSPEVQIAILTEQINNLNEHLRVHKKD…","""SAGGSPQVQIAILTEQINNLNEHLRVHKKD…",1.932703,2.674476,-0.741773,"""E7Q""",0.085098,false
"""GQTIHNGDQTYHYDNPEEAIKVAQKLAKIY…","""GQTIHNIDQTYHYDNPEEAIKVAQKLAKIY…",1.46799,1.529946,-0.061956,"""G7I""",0.007125,false
…,…,…,…,…,…,…,…
"""SAGGNQASVVANTLIPINTALTLVMMRSEV…","""SAGGNQASVVANTLIPINTALTLVMMRSEV…",5.973451,6.20524,-0.231789,"""S46W""",-0.026649,true
"""SAGGSAGTGIVNVSSSLNVRSSASTSSSVI…","""SAGGSAGTGIVNVSSSLNVRSSASTSSKVI…",2.993401,3.155036,-0.161636,"""K28S""",-0.018586,true
"""GGITGDVSAANKDAIRKQMDAAASKGDVET…","""GGITGDVSAANKDAIRKQMDAAASKGDVET…",1.23246,1.16441,0.06805,"""K39R""",0.019596,true


In [13]:
df["normalized_ddG_sigmoid"].describe()

statistic,value
str,f64
"""count""",416914.0
"""null_count""",0.0
"""mean""",-0.151919
"""std""",0.202927
"""min""",-0.96626
"""25%""",-0.252648
"""50%""",-0.106105
"""75%""",-0.020396
"""max""",0.986593


In [14]:
df.filter(pl.col('original_seq_full') == pl.col('mutated_seq_full'))

original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type,normalized_ddG_sigmoid,normalized_stability
str,str,f64,f64,f64,str,f64,f64


In [15]:
len(unique_seqs_lehner)

NameError: name 'unique_seqs_lehner' is not defined